# 17-01 捷径步长与想象回报的原理演示

对应第 17 章 Dreamer 4。依赖 NumPy、Matplotlib、PyTorch，CPU 即可。

这不是 Dreamer 4 复现：没有视频编码器、生成式 Transformer 或训练奖励模型。第一部分手工构造两个小步，说明大步监督怎么得到；第二部分用固定教学回报解释正负优势和行为先验约束。不能把结果当作 Dreamer 4 的生成或控制成绩。

## 1. 两小步到底怎样指导一大步

初始带噪分量为 0.2，总生成步长 d=0.5。两次小步方向分别假设为 0.8 与 0.4。它们在真实算法中由模型在不同中间点计算，本例将其固定，是为了逐项检查算术。横轴是生成进度，不是环境帧数。

In [ ]:
import numpy as np  # 计算生成步骤与轨迹回报
import matplotlib.pyplot as plt  # 显示步骤与概率变化
import torch  # 用梯度演示一次简化策略更新
tau = 0.0  # 指定当前生成进度
d = 0.5  # 指定一次大步覆盖的进度区间
x = 0.2  # 指定当前带噪表示的一个分量
first_direction = 0.8  # 假设模型在起点预测的小步方向
midpoint = x + (d / 2) * first_direction  # 按第一小步到达中间点
second_direction = 0.4  # 假设模型在中间点重新查询得到的方向
endpoint = midpoint + (d / 2) * second_direction  # 按第二小步到达区间终点
target_direction = (first_direction + second_direction) / 2  # 构造一次大步应学习的平均方向
big_endpoint = x + d * target_direction  # 检查大步目标对应的终点
fig, ax = plt.subplots(figsize=(7, 3.5))  # 创建生成进度示意图
ax.plot([tau, tau+d/2, tau+d], [x, midpoint, endpoint], "o-", label="Two half steps")  # 显示两次查询之间的中间状态
ax.plot([tau, tau+d], [x, big_endpoint], "s--", label="One-step target", markersize=7)  # 显示大步学习目标而非已训练网络输出
ax.set(xlabel="Generation progress", ylabel="One latent component")  # 防止把潜表示变化误读为机器人速度
ax.legend()  # 区分两小步与大步目标
plt.show()  # 显示它们的终点关系
print(f"中间点={midpoint:.2f}；两步终点={endpoint:.2f}；大步方向目标={target_direction:.2f}")  # 输出每项算术结果
assert np.isclose(endpoint, big_endpoint)  # 核对构造的监督目标与两小步一致


### 这张图证明了什么，没证明什么

实线在进度 0.25 处重新查询方向；虚线跳过中间查询，但它使用的是构造出来的理想监督目标。两者终点重合是算术设计的结果，不是网络已经学会捷径。

真正训练还需要：不同样本、不同进度、不同步长、停止梯度的目标，以及与真实数据相连的小步监督。单靠这张图不能推断某个网络能用几步生成高质量视频。

## 2. 从一段想象得到回报与优势

设两种候选动作导致的教学奖励分别为 [0.2, 0.4] 与 [0.1, -0.1]，末端价值分别为 0.5 与 0.0。折扣 0.9 表示更远奖励稍微降低权重；起点价值估计为 0.7。这里用简单两步回报，不实现论文的完整 λ 回报。

In [ ]:
rewards = np.array([[0.2, 0.4], [0.1, -0.1]])  # 给出两条想象轨迹的教学奖励
tail_values = np.array([0.5, 0.0])  # 给出未继续展开的更远未来价值
gamma = 0.9  # 设置未来奖励的折扣系数
returns = rewards[:, 0] + gamma * rewards[:, 1] + gamma**2 * tail_values  # 汇总两步奖励与尾部价值
baseline = 0.7  # 指定当前起点的价值估计
advantages = returns - baseline  # 比较每条轨迹是否好于当前预期
fig, ax = plt.subplots(figsize=(6, 3))  # 创建回报与基准对照图
ax.bar(["A", "B"], returns, color=["#497f99", "#b67d4c"])  # 显示两条轨迹的累计回报
ax.axhline(baseline, color="black", linestyle="--", label="Current value")  # 用虚线表示起点价值估计
ax.set(ylabel="Illustrative return")  # 明确这些是教学数值而非真实评测
ax.legend()  # 标明价值基准
plt.show()  # 显示回报与价值之间的相对关系
print("回报：", returns, "；优势：", advantages)  # 打印用于策略更新的具体数值


### 优势不是动作速度，也不是概率

柱子高于黑线，优势为正，说明这次想象比当前预期更好；低于黑线则为负。A 的回报是 0.965，优势 0.265。B 的回报是 0.01，优势 -0.69。它们都来自本例设定的奖励，不是真实环境反馈。

Dreamer 4 用学习到的奖励与价值计算训练信号。若这些预测错了，优势也可能误导策略，所以还需要行为先验限制和独立真实评测。

## 3. 概率怎样向更好的动作移动

这里只在同一个教学状态下设置两种动作。冻结的初始策略各给一半概率；可学习策略起初与它相同。用正负优势分别鼓励或抑制动作，再加入与初始分布的 KL 约束。这个两动作例子只展示论文策略目标的方向，不实现完整 PMPO 训练系统。

In [ ]:
logits = torch.zeros(2, requires_grad=True)  # 用两个可学习数控制当前状态的动作概率
prior = torch.tensor([0.5, 0.5])  # 固定进入想象训练前的行为先验
optimizer = torch.optim.SGD([logits], lr=0.2)  # 设置教学用概率更新步长
history = []  # 保存更新过程中两种动作的概率
for update in range(25):  # 多次使用同一教学反馈观察更新方向
    log_probs = torch.log_softmax(logits, dim=0)  # 将可学习分数变成对数概率
    probs = log_probs.exp()  # 得到真正归一化的动作概率
    kl = (probs * (log_probs - prior.log())).sum()  # 衡量当前策略偏离冻结先验的程度
    loss = -0.5 * log_probs[0] + 0.5 * log_probs[1] + 0.3 * kl  # 提高正优势动作概率并抑制负优势动作
    history.append(probs.detach().numpy().copy())  # 保存更新前的真实概率
    optimizer.zero_grad()  # 清除上一次的梯度
    loss.backward()  # 计算损失对动作分数的梯度
    optimizer.step()  # 更新动作分数而不更新任何世界模型
history = np.asarray(history)  # 整理记录供绘图
fig, ax = plt.subplots(figsize=(7, 3))  # 创建简化策略更新曲线
ax.plot(history[:, 0], label="A: positive advantage")  # 显示正优势动作的概率
ax.plot(history[:, 1], label="B: negative advantage")  # 显示负优势动作的概率
ax.set(xlabel="Illustrative update", ylabel="Probability", ylim=(0, 1))  # 明确纵轴才是策略概率
ax.legend()  # 标明正负反馈对应关系
plt.show()  # 显示策略更新方向
assert np.allclose(history.sum(axis=1), 1)  # 检查每一步仍是合法概率分布
assert history[-1, 0] > history[0, 0]  # 检查正优势反馈确实提高了动作概率


### 最后一图要避免怎样的误读

A 的概率提高，说明反馈改变了策略参数；不是 CEM 在当前时刻筛选动作，也不是把世界模型训练成了更高奖励的样子。整个实验中没有更新世界模型。

同一批固定正负标签被重复使用，只是为了看清方向。真实训练要重新采样动作和轨迹，更新价值估计，重新判断优势。这里负反馈持续不变时，概率可能趋向极端；KL 是软约束而非不可越过的安全边界。

练习：交换 A 与 B 的正负反馈项，看概率变化是否反转。思考：如果奖励预测错误而导致正负标签颠倒，策略为何会越学越偏？这正是想象训练仍需真实评测的原因。